In [1]:
from spark_utils import SparkUtils
from pathlib import Path
import shutil
import pyspark.sql.functions as F

connectors = "org.apache.spark:spark-sql-kafka-0-10_2.13:3.5.0,org.neo4j:neo4j-connector-apache-spark_2.13:5.3.10_for_spark_3"

su = SparkUtils(
    app_name="ProyectoFinalStreaming", 
    master_url="local",
    spark_packages=connectors
)

spark = su.spark



:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.neo4j#neo4j-connector-apache-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-de23ebc6-e117-43e8-9270-51209fd1b8cf;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;3.5.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;3.5.0 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	f

In [2]:
columnas_payload = [
    ("user_id", "int"),
    ("game_id", "int"),
    ("action", "string"),
    ("timestamp", "long"),
    ("session_duration_sec", "int")
]
schema = SparkUtils.generate_schema(columnas_payload)
#Conexión Kafka
kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9093") \
    .option("subscribe", "proyecto_streaming") \
    .option("startingOffsets", "latest") \
    .load()

#JSON Parse
parsed_df = kafka_df \
    .selectExpr("CAST(value AS STRING) as json_payload") \
    .withColumn("data", F.from_json(F.col("json_payload"), schema)) \
    .select("data.*")

print("Lectura de stream configurada correctamente.")

Lectura de stream configurada correctamente.


In [ ]:
def procesar_microbatch_neo4j(df_batch, batch_id):
    """Escribe cada micro-batch en Neo4j utilizando MERGE y CREATE."""
    
    cypher_merge_query = """
        MERGE (u:User {id: event.user_id})
        MERGE (g:Game {id: event.game_id})
        CREATE (u)-[r:INTERACTED_WITH {
            action: event.action,
            timestamp: event.timestamp,
            duration_sec: event.session_duration_sec
        }]->(g)
    """
    
    # Usa "neo4j-iteso:7687" si corres en la red de Docker, o "localhost:7687" si expones puertos
    df_batch.write \
        .format("org.neo4j.spark.DataSource") \
        .mode("Append") \
        .option("url", "bolt://host.docker.internal:7687") \
        .option("authentication.basic.username", "neo4j") \
        .option("authentication.basic.password", "neo4j@1234") \
        .option("query", cypher_merge_query) \
        .save()

# Iniciar el streaming
query = parsed_df.writeStream \
    .outputMode("append") \
    .foreachBatch(procesar_microbatch_neo4j) \
    .start()

print("Ejecutando streaming hacia Neo4j...")
query.awaitTermination()

26/05/06 01:58:27 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-7a0f134a-817b-4d73-a6db-a49c812ada7a. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/06 01:58:27 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Ejecutando streaming hacia Neo4j...


26/05/06 01:58:27 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                